# Βήμα 3: Διερευνητική ανάλυση (EDA) και μοντέλα τιμής

Runtime: **CPU** αρκεί. Διαβάζει το `processed/listings_model.csv.gz` από το βήμα 2.

Όλα τα γραφήματα σώζονται στο `figures/` και οι πίνακες στο `tables/` μέσα στον φάκελο `thesis`, έτοιμα για τη διπλωματική.

**Ερευνητικά ερωτήματα**
1. Ποιοι παράγοντες επηρεάζουν την τιμή;
2. Πώς διαφέρει το συναίσθημα των κριτικών ανάμεσα στις περιοχές;
3. Προσθέτει το sentiment πληροφορία για την τιμή, πέρα από τα χαρακτηριστικά και τις βαθμολογίες με αστέρια;

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/thesis
!pip install -q shap

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

os.makedirs('figures', exist_ok=True)
os.makedirs('tables', exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')
REGION_ORDER = ['crete', 'athens', 'thessaloniki']
REGION_LABELS = {'crete': 'Κρήτη', 'athens': 'Αθήνα', 'thessaloniki': 'Θεσσαλονίκη'}

def save(fig, name):
    fig.savefig(f'figures/{name}.png', dpi=200, bbox_inches='tight')
    plt.show()

df = pd.read_csv('processed/listings_model.csv.gz')
df['region_label'] = df['region'].map(REGION_LABELS)
LABEL_ORDER = [REGION_LABELS[r] for r in REGION_ORDER]
print(df.shape)

# Μέρος Α: Διερευνητική ανάλυση
## Α1. Περιγραφικά στατιστικά ανά περιοχή

In [ ]:
desc_cols = ['price', 'accommodates', 'bedrooms', 'n_amenities', 'review_scores_rating',
             'number_of_reviews', 'sent_mean', 'sent_neg_share', 'host_is_superhost']
desc_cols = [c for c in desc_cols if c in df.columns]
desc = df.groupby('region_label')[desc_cols].agg(['mean', 'median', 'std']).T.round(2)
desc = desc[LABEL_ORDER]
desc.to_csv('tables/descriptives_by_region.csv')
desc

## Α2. Κατανομή τιμών

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(data=df, x='region_label', y='price', order=LABEL_ORDER, showfliers=False, ax=axes[0])
axes[0].set(xlabel='', ylabel='Τιμή ανά βράδυ (€)', title='Τιμή ανά περιοχή (χωρίς ακραίες τιμές)')
sns.kdeplot(data=df, x='log_price', hue='region_label', hue_order=LABEL_ORDER, common_norm=False, ax=axes[1])
axes[1].set(xlabel='log(τιμή)', ylabel='Πυκνότητα', title='Κατανομή λογαρίθμου τιμής')
save(fig, 'price_distribution')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(data=df, x='room_type', y='price', hue='region_label', hue_order=LABEL_ORDER,
            estimator=np.median, errorbar=None, ax=ax)
ax.set(xlabel='Τύπος καταλύματος', ylabel='Διάμεση τιμή (€)', title='Διάμεση τιμή ανά τύπο καταλύματος')
ax.legend(title='')
save(fig, 'price_by_room_type')

## Α3. Χάρτες τιμών

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, r in zip(axes, REGION_ORDER):
    d = df[df['region'] == r]
    sc = ax.scatter(d['longitude'], d['latitude'], c=d['log_price'], s=2, cmap='viridis', alpha=0.6)
    ax.set_title(REGION_LABELS[r]); ax.set_xticks([]); ax.set_yticks([])
    ax.set_aspect('equal', adjustable='datalim')
fig.colorbar(sc, ax=axes, shrink=0.8, label='log(τιμή)')
save(fig, 'price_maps')

## Α4. Sentiment ανά περιοχή (ερευνητικό ερώτημα 2)
Κρατάμε καταλύματα με τουλάχιστον 3 κριτικές, ώστε ο μέσος όρος να είναι αξιόπιστος.

In [ ]:
MIN_REVIEWS = 3
s = df[df['n_reviews_sent'] >= MIN_REVIEWS]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(data=s, x='region_label', y='sent_mean', order=LABEL_ORDER, showfliers=False, ax=axes[0])
axes[0].set(xlabel='', ylabel='Μέσο sentiment score', title='Μέσο sentiment ανά κατάλυμα')
sns.barplot(data=s, x='region_label', y='sent_neg_share', order=LABEL_ORDER, ax=axes[1])
axes[1].set(xlabel='', ylabel='Μέσο ποσοστό αρνητικών κριτικών', title='Αρνητικές κριτικές ανά περιοχή (με 95% CI)')
save(fig, 'sentiment_by_region')

# Στατιστικός έλεγχος: διαφέρει το sentiment ανάμεσα στις περιοχές;
groups = [s.loc[s['region'] == r, 'sent_mean'] for r in REGION_ORDER]
H, p = stats.kruskal(*groups)
n = sum(len(g) for g in groups)
eps2 = (H - len(groups) + 1) / (n - len(groups))  # μέγεθος επίδρασης (epsilon squared)
print(f'Kruskal-Wallis: H = {H:.1f}, p = {p:.3g}, epsilon² = {eps2:.3f}')

rows = []
for i in range(3):
    for j in range(i + 1, 3):
        a, b = groups[i], groups[j]
        U, pu = stats.mannwhitneyu(a, b)
        rows.append({'σύγκριση': f'{REGION_LABELS[REGION_ORDER[i]]} vs {REGION_LABELS[REGION_ORDER[j]]}',
                     'U': U, 'p (Bonferroni)': min(pu * 3, 1), 'διάμεσος 1': a.median(), 'διάμεσος 2': b.median()})
pairwise = pd.DataFrame(rows).round(4)
pairwise.to_csv('tables/sentiment_pairwise_tests.csv', index=False)
pairwise

## Α5. Έλεγχος εγκυρότητας: συμφωνεί το sentiment με τα αστέρια;
Αν το μοντέλο sentiment δουλεύει σωστά, πρέπει να συσχετίζεται θετικά με τη βαθμολογία που δίνουν οι ίδιοι οι επισκέπτες.

In [ ]:
v = s.dropna(subset=['review_scores_rating'])
rows = []
for r in REGION_ORDER + ['όλες']:
    d = v if r == 'όλες' else v[v['region'] == r]
    rho, p = stats.spearmanr(d['sent_mean'], d['review_scores_rating'])
    rows.append({'περιοχή': REGION_LABELS.get(r, r), 'N': len(d), 'Spearman rho': round(rho, 3), 'p': p})
validity = pd.DataFrame(rows)
validity.to_csv('tables/sentiment_validity.csv', index=False)
display(validity)

fig, ax = plt.subplots(figsize=(6.5, 5))
hb = ax.hexbin(v['review_scores_rating'], v['sent_mean'], gridsize=40, bins='log', cmap='Blues',
               extent=(v['review_scores_rating'].quantile(0.01), 5, -1, 1))
ax.set(xlabel='Βαθμολογία (αστέρια)', ylabel='Μέσο sentiment score', title='Sentiment έναντι βαθμολογίας')
fig.colorbar(hb, label='log(πλήθος)')
save(fig, 'sentiment_vs_rating')

## Α6. Συσχετίσεις με την τιμή

In [ ]:
num_cols = ['log_price', 'accommodates', 'bedrooms', 'beds', 'bathrooms_n', 'n_amenities',
            'has_pool', 'has_sea_view', 'has_ac', 'host_is_superhost', 'host_tenure_years',
            'calculated_host_listings_count', 'availability_365', 'number_of_reviews',
            'review_scores_rating', 'sent_mean', 'sent_neg_share']
num_cols = [c for c in num_cols if c in s.columns]
corr = s[num_cols].corr(method='spearman')
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, annot=True, fmt='.2f', annot_kws={'size': 7}, ax=ax)
ax.set_title('Συσχετίσεις Spearman (καταλύματα με ≥3 κριτικές)')
save(fig, 'correlation_heatmap')

# Μέρος Β: Μοντέλα τιμής (ερευνητικά ερωτήματα 1 και 3)

Τρεις εκδοχές, στο **ίδιο δείγμα** (καταλύματα με ≥3 κριτικές και βαθμολογία), ώστε η σύγκριση να είναι δίκαιη:
- **Α (Βασικό):** χαρακτηριστικά καταλύματος, παροχές, οικοδεσπότης, τοποθεσία
- **Β (+ Αστέρια):** Α + βαθμολογίες και πλήθος κριτικών
- **Γ (+ Sentiment):** Β + μέσο sentiment και ποσοστό αρνητικών κριτικών

Η ερώτηση είναι αν το Γ είναι καλύτερο από το Β, δηλαδή αν **το κείμενο των κριτικών λέει κάτι που δεν λένε ήδη τα αστέρια**. Εξαρτημένη μεταβλητή: log(τιμή).

In [ ]:
m = df[(df['n_reviews_sent'] >= MIN_REVIEWS) & df['review_scores_rating'].notna()].copy()

# Ομαδοποίηση σπάνιων property types
top_pt = m['property_type'].value_counts().index[:12]
m['property_type_g'] = np.where(m['property_type'].isin(top_pt), m['property_type'], 'Other')

# Λογαριθμικοί μετασχηματισμοί για πολύ ασύμμετρες μεταβλητές
m['log_min_nights'] = np.log1p(m['minimum_nights'].clip(upper=365))
m['log_host_listings'] = np.log1p(m['calculated_host_listings_count'])
m['log_n_reviews'] = np.log1p(m['number_of_reviews'])

BASE_NUM = ['accommodates', 'bedrooms', 'beds', 'bathrooms_n', 'bathroom_shared', 'n_amenities',
            'has_pool', 'has_ac', 'has_free_parking', 'has_sea_view', 'has_workspace', 'has_elevator',
            'has_balcony', 'has_self_checkin', 'host_is_superhost', 'host_response_rate',
            'host_tenure_years', 'log_host_listings', 'instant_bookable', 'log_min_nights',
            'availability_365', 'latitude', 'longitude']
STARS_NUM = ['review_scores_rating', 'review_scores_cleanliness', 'review_scores_location',
             'review_scores_value', 'log_n_reviews']
SENT_NUM = ['sent_mean', 'sent_neg_share']
# Η περιοχή καλύπτεται από τις γειτονιές (κάθε γειτονιά ανήκει σε μία περιοχή)
CAT = ['room_type', 'property_type_g', 'neighbourhood_cleansed']

BASE_NUM = [c for c in BASE_NUM if c in m.columns]
STARS_NUM = [c for c in STARS_NUM if c in m.columns]

def make_X(num_cols, verbose=False):
    X = m[num_cols].copy().replace([np.inf, -np.inf], np.nan)
    empty = [c for c in X.columns if X[c].isna().all()]
    if empty:
        if verbose:
            print('Αφαιρούνται στήλες χωρίς καμία τιμή:', empty)
        X = X.drop(columns=empty)
    for c in list(X.columns):  # συμπλήρωση ελλειπόντων με διάμεσο + δείκτης ελλείποντος
        if X[c].isna().any():
            X[c + '_missing'] = X[c].isna().astype(int)
            X[c] = X[c].fillna(X[c].median())
    dummies = pd.get_dummies(m[CAT].astype(str), drop_first=True, dtype=int)
    X = pd.concat([X, dummies], axis=1)
    assert np.isfinite(X.to_numpy(dtype=float)).all(), 'Υπάρχουν ακόμα NaN/inf'
    return X

_ = make_X(BASE_NUM + STARS_NUM + SENT_NUM, verbose=True)

SPECS = {
    'Α: Βασικό': BASE_NUM,
    'Β: + Αστέρια': BASE_NUM + STARS_NUM,
    'Γ: + Sentiment': BASE_NUM + STARS_NUM + SENT_NUM,
}
y = m['log_price'].values
print(f'Δείγμα μοντέλων: {len(m):,} καταλύματα')
print(m['region'].value_counts())

## Β1. Γραμμική παλινδρόμηση (hedonic pricing)
Ερμηνεύσιμοι συντελεστές, με robust τυπικά σφάλματα (HC1).

In [ ]:
import statsmodels.api as sm

ols_results = {}
for name, cols in SPECS.items():
    X = sm.add_constant(make_X(cols).astype(float))
    ols_results[name] = sm.OLS(y, X).fit(cov_type='HC1')
    r = ols_results[name]
    print(f'{name:18s} R² = {r.rsquared:.3f}   adj. R² = {r.rsquared_adj:.3f}   N = {int(r.nobs):,}')

# Συντελεστές του πλήρους μοντέλου για τις βασικές μεταβλητές
full = ols_results['Γ: + Sentiment']
show = [c for c in BASE_NUM + STARS_NUM + SENT_NUM if c in full.params.index and c not in ('latitude', 'longitude')]
coef = pd.DataFrame({'coef': full.params[show], 'std err': full.bse[show], 'p': full.pvalues[show]})
coef['% επίδραση'] = (np.exp(coef['coef']) - 1) * 100
coef = coef.round(4)
coef.to_csv('tables/ols_full_coefficients.csv')
coef

**Πώς διαβάζεται:** επειδή η εξαρτημένη είναι log(τιμή), η στήλη «% επίδραση» δείχνει κατά πόσο % αλλάζει η τιμή όταν η μεταβλητή αυξάνεται κατά 1 μονάδα, με όλα τα άλλα σταθερά. Για το `sent_mean` (κλίμακα -1 έως +1), μια πιο ρεαλιστική αλλαγή είναι 0,1, οπότε διαίρεσε περίπου με το 10.

In [ ]:
# Έλεγχος: βελτιώνει στατιστικά σημαντικά το Γ το Β; (F-test εμφωλευμένων μοντέλων, χωρίς robust SE)
Xb = sm.add_constant(make_X(SPECS['Β: + Αστέρια']).astype(float))
Xc = sm.add_constant(make_X(SPECS['Γ: + Sentiment']).astype(float))
rb, rc = sm.OLS(y, Xb).fit(), sm.OLS(y, Xc).fit()
f, pf, df_diff = rc.compare_f_test(rb)
print(f'F-test Γ έναντι Β: F = {f:.2f}, p = {pf:.3g}, επιπλέον μεταβλητές = {int(df_diff)}')

# Επίδραση του sentiment ανά περιοχή (το πλήρες μοντέλο χωριστά σε κάθε περιοχή)
rows = []
for r in REGION_ORDER:
    idx = (m['region'] == r).values
    Xr = make_X(SPECS['Γ: + Sentiment']).loc[idx]
    Xr = Xr.loc[:, Xr.nunique() > 1]
    res = sm.OLS(y[idx], sm.add_constant(Xr.astype(float))).fit(cov_type='HC1')
    for v in SENT_NUM:
        rows.append({'περιοχή': REGION_LABELS[r], 'μεταβλητή': v, 'coef': res.params[v],
                     'p': res.pvalues[v], 'R²': res.rsquared, 'N': int(res.nobs)})
by_region = pd.DataFrame(rows).round(4)
by_region.to_csv('tables/ols_sentiment_by_region.csv', index=False)
by_region

## Β2. Μοντέλα μηχανικής μάθησης (Random Forest, XGBoost)
5-fold cross-validation. Μετρικές σε log(τιμή): RMSE (όσο μικρότερο τόσο καλύτερο) και R². Παίρνει μερικά λεπτά.

In [ ]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

MODELS = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=150, min_samples_leaf=3, max_features=0.3, max_samples=0.5,
                                           n_jobs=-1, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8,
                            colsample_bytree=0.7, tree_method='hist', n_jobs=-1, random_state=42),
}
cv = KFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for spec, cols in SPECS.items():
    X = make_X(cols).astype(float)
    for mname, model in MODELS.items():
        sc = cross_validate(model, X, y, cv=cv, scoring=('neg_root_mean_squared_error', 'r2'))
        rows.append({'εκδοχή': spec, 'μοντέλο': mname,
                     'RMSE': -sc['test_neg_root_mean_squared_error'].mean(),
                     'RMSE sd': sc['test_neg_root_mean_squared_error'].std(),
                     'R²': sc['test_r2'].mean()})
        print(f'{spec:18s} {mname:18s} RMSE = {rows[-1]["RMSE"]:.4f}  R² = {rows[-1]["R²"]:.4f}')

cv_results = pd.DataFrame(rows).round(4)
cv_results.to_csv('tables/model_comparison_cv.csv', index=False)
cv_results.pivot(index='μοντέλο', columns='εκδοχή', values='R²')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=cv_results, x='μοντέλο', y='R²', hue='εκδοχή', ax=ax)
ax.set(xlabel='', ylabel='R² (5-fold CV)', title='Απόδοση μοντέλων ανά εκδοχή χαρακτηριστικών')
ax.set_ylim(cv_results['R²'].min() - 0.05, cv_results['R²'].max() + 0.02)
ax.legend(title='', loc='lower right')
save(fig, 'model_comparison')

## Β3. Ερμηνεία με SHAP (XGBoost, πλήρες μοντέλο)
Ποιες μεταβλητές «οδηγούν» τις προβλέψεις και προς ποια κατεύθυνση.

In [ ]:
import shap

X_full = make_X(SPECS['Γ: + Sentiment']).astype(float)
xgb = MODELS['XGBoost'].fit(X_full, y)
X_sample = X_full.sample(min(3000, len(X_full)), random_state=42)
shap_values = shap.TreeExplainer(xgb).shap_values(X_sample)

plt.figure()
shap.summary_plot(shap_values, X_sample, max_display=20, show=False)
plt.title('SHAP: επίδραση μεταβλητών στην log(τιμή)')
save(plt.gcf(), 'shap_summary')

imp = pd.Series(np.abs(shap_values).mean(axis=0), index=X_sample.columns).sort_values(ascending=False)
imp.head(25).round(4).to_csv('tables/shap_importance.csv')
print('Θέση των μεταβλητών sentiment στη σειρά σπουδαιότητας:')
for v in SENT_NUM:
    print(f'  {v}: #{list(imp.index).index(v) + 1} από {len(imp)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, v in zip(axes, SENT_NUM):
    j = list(X_sample.columns).index(v)
    ax.scatter(X_sample[v], shap_values[:, j], s=4, alpha=0.4)
    ax.axhline(0, color='grey', lw=0.8)
    ax.set(xlabel=v, ylabel='SHAP value (επίδραση στη log τιμή)')
fig.suptitle('Πώς επηρεάζει το sentiment την προβλεπόμενη τιμή')
save(fig, 'shap_sentiment_dependence')

## Τι να κρατήσεις για τη διπλωματική
- `tables/model_comparison_cv.csv`: η βασική απάντηση στο ερευνητικό ερώτημα 3 (Γ έναντι Β)
- `tables/ols_full_coefficients.csv` και `ols_sentiment_by_region.csv`: ερμηνεύσιμες επιδράσεις
- `tables/sentiment_pairwise_tests.csv`, `sentiment_validity.csv`: ερώτημα 2 και έλεγχος εγκυρότητας
- `figures/`: όλα τα γραφήματα σε 200 dpi